## Raw EEG Temporal Ablation Analysis 2

The goal of this notebook is to apply temporal ablation to determine the time windows of raw EEG data that is best features for a classifier to identify MDD in patients.

The classifier being used will be Logisitic Regression with GroupKFold cross-validation.

## 1. Extracting the raw eeg data

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import mne

from pathlib import Path 

from sklearn.model_selection import GroupKFold, cross_validate, cross_val_predict
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    roc_auc_score,
    confusion_matrix, 
    ConfusionMatrixDisplay
)
from sklearn.base import clone

import os 

In [2]:
def find_project_root():
    p = Path.cwd()
    for parent in [p, *p.parents]:
        if (parent / ".git").exists():
            return parent
    raise FileNotFoundError("Project root (with .git) not found")

PROJECT_ROOT = find_project_root()
RAW_DATA_DIR = PROJECT_ROOT / "data" / "raw" / "nm000114"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed_features"
RESULTS_DIR = PROJECT_ROOT / "results"
SEGMENTED_DATA_DIR = PROJECT_ROOT / "data" / "segmented_data"

def get_eeg_file(subject_id: str, condition: str):
    """ Get the path to the EEG file for a given subject and condition.
    """
    return RAW_DATA_DIR / subject_id / "eeg" / f"{subject_id}_task-{condition}_eeg.edf"

def infer_label_from_subject(subject_id: str) -> int:
    """
    Infer the label (0 for healthy control, 1 for MDD) from the subject ID.
    """
    subject_upper = subject_id.upper()
    if "HS" in subject_upper:
        return 0
    elif "MDD" in subject_upper:
        return 1 
    else: 
        raise ValueError(f"Could not infer label from subject ID: {subject_id}")
    
def parse_condition_from_filename(filepath: Path) -> str:
    """
    Parse the condition (eyesClosed, eyesOpen, P300) from the filename.
    """
    name = filepath.name 
    if "task-eyesClosed" in name:
        return "eyesClosed"
    elif "task-eyesOpen" in name:
        return "eyesOpen"
    elif "task-P300" in name:
        return "P300"
    else:
        raise ValueError(f"Could not parse condition from filename: {name}")
        
edf_files =sorted(RAW_DATA_DIR.glob("sub*/eeg/*.edf"))
        
rows = []
for filepath in edf_files:
    subject_id = filepath.parts[-3] 
    rows.append({
        "patient_id": subject_id,
        "recording_id": filepath.stem,
        "label": infer_label_from_subject(subject_id),
        "condition": parse_condition_from_filename(filepath),
        "filepath": str(filepath),
    })


# Create a DataFrame from the metadata
metadata_df = pd.DataFrame(rows)

COMMON_CHANNELS = [
'EEG Fp1-LE', 'EEG F3-LE', 'EEG C3-LE', 'EEG P3-LE', 'EEG O1-LE',
 'EEG F7-LE', 'EEG T3-LE', 'EEG T5-LE', 'EEG Fz-LE', 'EEG Fp2-LE', 
 'EEG F4-LE', 'EEG C4-LE', 'EEG P4-LE', 'EEG O2-LE', 'EEG F8-LE', 
 'EEG T4-LE', 'EEG T6-LE', 'EEG Cz-LE', 'EEG Pz-LE', 'EEG A2-A1'
]


## 2. Identifying the Time points

In [3]:
metadata_df.head()

,patient_id,recording_id,label,condition,filepath
0,sub-HS1,sub-HS1_task-P300_eeg,0,P300,/home/srlee185/MSSE-277b-final-project-/data/r...
1,sub-HS1,sub-HS1_task-eyesClosed_eeg,0,eyesClosed,/home/srlee185/MSSE-277b-final-project-/data/r...
2,sub-HS1,sub-HS1_task-eyesOpen_eeg,0,eyesOpen,/home/srlee185/MSSE-277b-final-project-/data/r...
3,sub-HS10,sub-HS10_task-P300_eeg,0,P300,/home/srlee185/MSSE-277b-final-project-/data/r...
4,sub-HS10,sub-HS10_task-eyesClosed_eeg,0,eyesClosed,/home/srlee185/MSSE-277b-final-project-/data/r...


In [4]:
# Seeing number of time points across datapoints
conditions = []
timepoints = []

for _, row in metadata_df.iterrows(): 
    filepath = row['filepath']
    condition = row['condition']
    raw = mne.io.read_raw_edf(filepath, preload=True, verbose=False)
    raw.pick(COMMON_CHANNELS)

    conditions.append(condition)
    timepoints.append(raw.get_data().shape[1])

timepoints_df = pd.DataFrame({
    "condition": conditions,
    "n_timepoints": timepoints
})

closed_tp = timepoints_df[timepoints_df['condition'] == 'eyesClosed']['n_timepoints'].value_counts()
p300_tp = timepoints_df[timepoints_df['condition'] == 'P300']['n_timepoints'].value_counts()
open_tp = timepoints_df[timepoints_df['condition'] == 'eyesOpen']['n_timepoints'].value_counts()

print(f"Num TP for eyesClosed: {closed_tp}")
print(f"Num TP for eyesOpen: {open_tp}")
print(f"Num TP for P300: {p300_tp}")


Num TP for eyesClosed: n_timepoints
76800    17
77056    12
77312     8
76544     5
77568     4
76032     3
76288     2
96256     1
74752     1
79104     1
78592     1
91136     1
46080     1
61696     1
Name: count, dtype: int64
Num TP for eyesOpen: n_timepoints
76800    14
77312    11
77056    10
76544     6
77568     3
78080     3
76032     3
76288     2
75776     2
89856     1
49664     1
81920     1
75520     1
79872     1
48384     1
79616     1
78592     1
Name: count, dtype: int64
Num TP for P300: n_timepoints
154880    10
162048     5
161536     5
161792     4
156160     3
155392     3
162816     3
155648     2
164864     2
156416     2
163072     2
160768     2
168448     2
162304     1
154624     1
158720     1
164608     1
156672     1
155904     1
153856     1
175616     1
166144     1
162560     1
160512     1
163328     1
164096     1
161024     1
157184     1
163584     1
Name: count, dtype: int64


In [29]:
# Testing time windows on one file

i = 0
filepath = metadata_df.iloc[i, -1]
condition = metadata_df.iloc[i, -2]

raw = mne.io.read_raw_edf(filepath, preload=True, verbose=False)
raw.pick(COMMON_CHANNELS)

# To get the length of recording in seconds = time points / sampling frequency 
length_of_recording = raw.n_times / raw.info['sfreq'] 

# Divide into time windows
time_windows = mne.make_fixed_length_epochs(
    raw,
    duration = length_of_recording / 10, # size of time windown in seconds
    preload = True,
    overlap = 0.0 # overlap size between windows
)

print(time_windows.get_data().shape) # should be 10 epochs, 20 channels, total timepoints / 10
print(time_windows.tmin, time_windows.tmax)


Not setting metadata
10 matching events found
No baseline correction applied
0 projection items activated
Using data from preloaded Raw for 10 events and 15488 original time points ...
0 bad epochs dropped
(10, 20, 15488)
0.0 60.49609375


In [36]:
# Writing a function for extracting time window

def extract_time_windows(filepath, window_time):
    """
    Divides raw EDFs into certain n_windows time windows.
    
    Returns eeg file in the number of (window_time windows), and readings per window
    """

    raw = mne.io.read_raw_edf(filepath, preload=True, verbose=False)
    raw.pick(COMMON_CHANNELS)

    epochs = mne.make_fixed_length_epochs(
        raw,
        duration = window_time,
        preload = True,
        overlap = 0.0,
    ) 

    data = epochs.get_data()

    # Adding the remainder window
    sfreq = raw.info['sfreq']
    samp_window = int(window_time * sfreq)
    total_tp = raw.n_times
    r = total_tp % samp_window

    if r > 0:
        remainder = raw.get_data()[:, -r:]
        pad_width = samp_window - r
        padded = np.pad(
            remainder,
            pad_width=((0,0), (0, pad_width)),
            mode = 'constant'
        )
        padded = padded[np.newaxis, ...]

        data = np.concatenate([data, padded], axis=0)


    return data.reshape(data.shape[0], -1)

In [38]:
extract_time_windows(filepath, 10)

Not setting metadata
60 matching events found
No baseline correction applied
0 projection items activated
Using data from preloaded Raw for 60 events and 2560 original time points ...
0 bad epochs dropped


array([[-1.58512093e-05, -1.84514076e-05, -1.47511254e-05, ...,
         1.85014115e-06,  2.50019074e-07,  6.45049210e-06],
       [ 1.13508659e-05,  2.39518273e-05,  3.75528649e-05, ...,
         4.35033188e-06,  9.85075151e-06,  1.23509422e-05],
       [ 1.05008011e-06, -2.95022507e-06, -1.08508278e-05, ...,
        -3.85029374e-06, -1.16508888e-05, -1.53511711e-05],
       ...,
       [ 4.75036240e-06,  1.24509499e-05,  1.07508202e-05, ...,
         1.55511864e-05,  1.47511254e-05,  7.55057603e-06],
       [ 2.95522545e-05,  2.79521324e-05,  1.89514458e-05, ...,
         2.50019074e-07, -2.75020981e-06, -9.55072862e-06],
       [ 2.67520409e-05,  1.98515145e-05,  1.31510033e-05, ...,
         0.00000000e+00,  0.00000000e+00,  0.00000000e+00]],
      shape=(61, 51200))

## 3. Training a model based on time interval windows on EEG data

In [8]:
# Baseline augmentation
gfk = GroupKFold(n_splits=5)

pipeline_svm = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", SVC(kernel="rbf",probability=True)
     )
])

print("Most Common TP for each condition")
print(f"Closed TP: {closed_tp.idxmax()} \n Open TP: {open_tp.idxmax()} \n P300 TP: {p300_tp.idxmax()}")

Most Common TP for each condition
Closed TP: 76800 
 Open TP: 76800 
 P300 TP: 154880


In [44]:
# Testing pipeline with the first 10 second window

seconds = 10
X_list = []
y = []
groups = []

for _, row in metadata_df.iterrows():
    diagnosis = row['label']
    filepath = row['filepath']
    group = row['patient_id']

    window_data = extract_time_windows(filepath, seconds)[0]
    X_list.append(window_data)
    y.append(diagnosis)
    groups.append(group)

X = np.vstack(X_list)

Not setting metadata
60 matching events found
No baseline correction applied
0 projection items activated
Using data from preloaded Raw for 60 events and 2560 original time points ...
0 bad epochs dropped
Not setting metadata
30 matching events found
No baseline correction applied
0 projection items activated
Using data from preloaded Raw for 30 events and 2560 original time points ...
0 bad epochs dropped
Not setting metadata
35 matching events found
No baseline correction applied
0 projection items activated
Using data from preloaded Raw for 35 events and 2560 original time points ...
0 bad epochs dropped
Not setting metadata
60 matching events found
No baseline correction applied
0 projection items activated
Using data from preloaded Raw for 60 events and 2560 original time points ...
0 bad epochs dropped
Not setting metadata
37 matching events found
No baseline correction applied
0 projection items activated
Using data from preloaded Raw for 37 events and 2560 original time points 

In [ ]:
results_svm = cross_validate(
    pipeline_svm,
    X,
    y,
    cv=gfk.split(X, y, groups),
    scoring=["accuracy", "f1", "roc_auc"]
)

print("Accuracy:", results_svm["test_accuracy"])
print("Mean accuracy:", results_svm["test_accuracy"].mean())
print("Std accuracy:", results_svm["test_accuracy"].std())

print("F1:", results_svm["test_f1"])
print("Mean F1:", results_svm["test_f1"].mean())

print("ROC-AUC:", results_svm["test_roc_auc"])
print("Mean ROC-AUC:", results_svm["test_roc_auc"].mean())
